In [4]:
from ultralytics import YOLO

In [11]:
model = YOLO("yolo26m.pt")  # YOLO26n is supported by Ultralytics
results = model.track(
    source="Houston, we Have an Overhead!  8 Ball Match Play!.mp4",
    tracker="bytetrack.yaml",
    persist=True,
    conf=0.25,
    save=True
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/150) /home/asura/repos/DataDrivenAndAgentBasedModellingBMETEFTBsPAAMD-00/cls09/Houston, we Have an Overhead!  8 Ball Match Play!.mp4: 384x640 1 person, 245.8ms
video 1/1 (frame 2/150) /home/asura/repos/DataDrivenAndAgentBasedModellingBMETEFTBsPAAMD-00/cls09/Houston, we Have an Overhead!  8 Ball Match Play!.mp4: 384x640 1 person, 221.2ms
video 1/1 (frame 3/150) /home/asura/repos/DataDrivenAndAgentBasedModellingBMETEFTBsPAAMD-00/cls09/Ho

In [ ]:
import cv2
import numpy as np
import sys
from scipy.optimize import linear_sum_assignment


VIDEO = "Houston, we Have an Overhead!  8 Ball Match Play!.mp4"
OUT = "tracked_hough.mp4"


class Track:
    def __init__(self, xy, radius, hist, track_id):
        self.id = track_id
        self.kf = cv2.KalmanFilter(4, 2)
        self.kf.transitionMatrix = np.array([
            [1, 0, 1, 0],
            [0, 1, 0, 1],
            [0, 0, 1, 0],
            [0, 0, 0, 1],
        ], np.float32)
        self.kf.measurementMatrix = np.array([
            [1, 0, 0, 0],
            [0, 1, 0, 0],
        ], np.float32)

        self.kf.processNoiseCov = np.eye(4, dtype=np.float32) * 0.03
        self.kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * 3.0
        self.kf.errorCovPost = np.eye(4, dtype=np.float32)

        x, y = xy
        self.kf.statePost = np.array([[x], [y], [0], [0]], np.float32)

        self.missed = 0
        self.trace = [(float(x), float(y))]
        self.radius = radius
        self.hist = hist
        self.last_velocity = np.array([0.0, 0.0])        

    def predict(self):
        p = self.kf.predict()
        return np.array([p[0, 0], p[1, 0]])

    def update(self, xy, radius, hist):
        old_pos = np.array(self.trace[-1])
        new_pos = np.array(xy)

        self.last_velocity = new_pos - old_pos
        self.radius = 0.8 * self.radius + 0.2 * radius

        if hist is not None:
            if self.hist is None:
                self.hist = hist
            else:
                self.hist = 0.9 * self.hist + 0.1 * hist

        self.kf.correct(np.array([[np.float32(xy[0])], [np.float32(xy[1])]]))
        self.missed = 0
        self.trace.append((float(xy[0]), float(xy[1])))

    def mark_missed(self):
        self.missed += 1
        p = self.predict()
        self.trace.append((float(p[0]), float(p[1])))


def detect_circles(frame):
    """
    Tune these values for your video.
    For best results, crop/warp to the table before calling this.
    """
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Helps suppress cloth texture/noise
    blur = cv2.medianBlur(gray, 5)

    circles = cv2.HoughCircles(
        blur,
        cv2.HOUGH_GRADIENT,
        dp=1.2,
        minDist=22,
        param1=80,
        param2=18,
        minRadius=6,
        maxRadius=22,
    )

    detections = []

    if circles is not None:
        circles = np.round(circles[0]).astype(int)

        for x, y, r in circles:
            # Optional: reject detections near borders
            if x < 5 or y < 5 or x >= frame.shape[1] - 5 or y >= frame.shape[0] - 5:
                continue

            detections.append((float(x), float(y), float(r)))

    return detections


def assign_tracks(tracks, detections, frame, max_cost=0.75):
    if not tracks:
        return [], list(range(len(detections))), []

    if not detections:
        return [], [], list(range(len(tracks)))

    cost = np.zeros((len(tracks), len(detections)), dtype=np.float32)

    det_hists = []
    for x, y, r in detections:
        det_hists.append(circle_histogram(frame, x, y, r))

    for i, t in enumerate(tracks):
        pred = t.predict()

        for j, (x, y, r) in enumerate(detections):
            det_pos = np.array([x, y])

            position_cost = np.linalg.norm(pred - det_pos) / 60.0
            position_cost = min(position_cost, 1.0)

            color_cost = hist_cost(t.hist, det_hists[j])

            radius_cost = abs(t.radius - r) / max(t.radius, r, 1.0)
            radius_cost = min(radius_cost, 1.0)

            expected_pos = np.array(t.trace[-1]) + t.last_velocity
            velocity_cost = np.linalg.norm(expected_pos - det_pos) / 80.0
            velocity_cost = min(velocity_cost, 1.0)

            cost[i, j] = (
                0.55 * position_cost +
                0.30 * color_cost +
                0.10 * radius_cost +
                0.05 * velocity_cost
            )

    row_ind, col_ind = linear_sum_assignment(cost)

    matches = []
    unmatched_dets = set(range(len(detections)))
    unmatched_tracks = set(range(len(tracks)))

    for r, c in zip(row_ind, col_ind):
        if cost[r, c] <= max_cost:
            matches.append((r, c, det_hists[c]))
            unmatched_tracks.discard(r)
            unmatched_dets.discard(c)

    return matches, list(unmatched_dets), list(unmatched_tracks)


def draw_tracks(frame, tracks):
    for t in tracks:
        if len(t.trace) < 2:
            continue

        pts = np.array(t.trace[-80:], dtype=np.int32)

        for i in range(1, len(pts)):
            cv2.line(frame, tuple(pts[i - 1]), tuple(pts[i]), (0, 255, 255), 2)

        x, y = pts[-1]
        cv2.circle(frame, (x, y), 5, (0, 0, 255), -1)
        cv2.putText(
            frame,
            f"ID {t.id}",
            (x + 8, y - 8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (255, 255, 255),
            1,
        )

def circle_histogram(frame, x, y, r):
    h, w = frame.shape[:2]

    x, y, r = int(x), int(y), int(r)
    x1, x2 = max(0, x-r), min(w, x+r)
    y1, y2 = max(0, y-r), min(h, y+r)

    patch = frame[y1:y2, x1:x2]
    if patch.size == 0:
        return None

    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)

    mask = np.zeros((y2-y1, x2-x1), dtype=np.uint8)
    cv2.circle(mask, (x-x1, y-y1), r, 255, -1)

    hist = cv2.calcHist(
        [hsv],
        [0, 1],
        mask,
        [24, 16],
        [0, 180, 0, 256]
    )

    cv2.normalize(hist, hist)
    return hist

def hist_cost(h1, h2):
    if h1 is None or h2 is None:
        return 1.0

    similarity = cv2.compareHist(h1, h2, cv2.HISTCMP_CORREL)
    return 1.0 - max(0.0, similarity)

cap = cv2.VideoCapture(VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer = cv2.VideoWriter(
    OUT,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w, h),
)

tracks = []
next_id = 0
max_missed = 12

frame_idx = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    detections = detect_circles(frame)

    matches, unmatched_dets, unmatched_tracks = assign_tracks(
        tracks, detections, frame
    )

    for ti, di, appearance in matches:
        x, y, r = detections[di]
        tracks[ti].update((x, y), r, appearance)

    for di in unmatched_dets:
        x, y, r = detections[di]
        appearance = circle_histogram(frame, x, y, r)
        tracks.append(Track((x, y), r, appearance, next_id))
        next_id += 1

    tracks = [t for t in tracks if t.missed <= max_missed]

    # Draw raw circle detections
    for x, y, r in detections:
        cv2.circle(frame, (int(x), int(y)), int(r), (0, 255, 0), 2)

    draw_tracks(frame, tracks)

    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()

print(f"Saved: {OUT}")

Saved: tracked_hough.mp4


In [19]:
import cv2
import numpy as np
import sys
from scipy.optimize import linear_sum_assignment


VIDEO = 'Houston, we Have an Overhead!  8 Ball Match Play!.mp4'
OUT = "tracked_hough_v2.mp4"


# ----------------------------
# Tunable parameters
# ----------------------------

MIN_RADIUS = 6
MAX_RADIUS = 22
MIN_DIST = 22

HOUGH_PARAM1 = 80
HOUGH_PARAM2 = 18

TABLE_MARGIN = 150

MIN_BALL_QUALITY = 0.55
MIN_ASSIGN_QUALITY = 0.45

MAX_MISSED = 15
PENDING_HITS_TO_PROMOTE = 3
PENDING_MAX_AGE = 5

BASE_MAX_JUMP = 35
VELOCITY_JUMP_FACTOR = 2.5
JUMP_EXTRA = 20

MAX_ASSIGN_COST = 0.75

TRACE_LENGTH = 100


# ----------------------------
# Detection helpers
# ----------------------------

def inside_table(x, y, r, frame_shape, margin=35):
    h, w = frame_shape[:2]
    return (
        x - r > margin and
        x + r < w - margin and
        y - r > margin and
        y + r < h - margin
    )


def ball_quality_score(frame, x, y, r):
    h, w = frame.shape[:2]
    x, y, r = int(x), int(y), int(r)

    x1, x2 = max(0, x - r), min(w, x + r)
    y1, y2 = max(0, y - r), min(h, y + r)

    patch = frame[y1:y2, x1:x2]
    if patch.size == 0:
        return 0.0

    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)

    mask = np.zeros(gray.shape, dtype=np.uint8)
    cx = x - x1
    cy = y - y1

    safe_r = min(
        r,
        cx,
        cy,
        mask.shape[1] - cx - 1,
        mask.shape[0] - cy - 1
    )

    if safe_r <= 2:
        return 0.0

    cv2.circle(mask, (cx, cy), safe_r, 255, -1)

    mean_sat = cv2.mean(hsv[:, :, 1], mask=mask)[0]
    std_gray = cv2.meanStdDev(gray, mask=mask)[1][0, 0]

    edges = cv2.Canny(gray, 50, 120)
    edge_pixels = cv2.countNonZero(cv2.bitwise_and(edges, edges, mask=mask))
    mask_pixels = max(cv2.countNonZero(mask), 1)
    edge_density = edge_pixels / mask_pixels

    score = 0.0

    if std_gray > 8:
        score += 0.35
    if edge_density > 0.04:
        score += 0.35
    if mean_sat > 20:
        score += 0.30

    return min(score, 1.0)


def circle_histogram(frame, x, y, r):
    h, w = frame.shape[:2]
    x, y, r = int(x), int(y), int(r)

    x1, x2 = max(0, x - r), min(w, x + r)
    y1, y2 = max(0, y - r), min(h, y + r)

    patch = frame[y1:y2, x1:x2]
    if patch.size == 0:
        return None

    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)

    mask = np.zeros((y2 - y1, x2 - x1), dtype=np.uint8)
    cx = x - x1
    cy = y - y1

    safe_r = min(
        r,
        cx,
        cy,
        mask.shape[1] - cx - 1,
        mask.shape[0] - cy - 1
    )

    if safe_r <= 2:
        return None

    cv2.circle(mask, (cx, cy), safe_r, 255, -1)

    hist = cv2.calcHist(
        [hsv],
        [0, 1],
        mask,
        [24, 16],
        [0, 180, 0, 256]
    )

    cv2.normalize(hist, hist)
    return hist


def hist_cost(h1, h2):
    if h1 is None or h2 is None:
        return 1.0

    similarity = cv2.compareHist(h1, h2, cv2.HISTCMP_CORREL)

    if np.isnan(similarity):
        return 1.0

    return 1.0 - max(0.0, min(1.0, similarity))


def detect_circles(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.medianBlur(gray, 5)

    circles = cv2.HoughCircles(
        blur,
        cv2.HOUGH_GRADIENT,
        dp=1.2,
        minDist=MIN_DIST,
        param1=HOUGH_PARAM1,
        param2=HOUGH_PARAM2,
        minRadius=MIN_RADIUS,
        maxRadius=MAX_RADIUS,
    )

    detections = []

    if circles is None:
        return detections

    circles = np.round(circles[0]).astype(int)

    for x, y, r in circles:
        if not inside_table(x, y, r, frame.shape, TABLE_MARGIN):
            continue

        q = ball_quality_score(frame, x, y, r)

        if q < MIN_BALL_QUALITY:
            continue

        detections.append({
            "x": float(x),
            "y": float(y),
            "r": float(r),
            "quality": float(q),
            "hist": circle_histogram(frame, x, y, r)
        })

    return detections


# ----------------------------
# Kalman track
# ----------------------------

class Track:
    def __init__(self, xy, radius, hist, quality, track_id):
        self.id = track_id

        self.kf = cv2.KalmanFilter(4, 2)

        self.kf.transitionMatrix = np.array([
            [1, 0, 1, 0],
            [0, 1, 0, 1],
            [0, 0, 1, 0],
            [0, 0, 0, 1],
        ], np.float32)

        self.kf.measurementMatrix = np.array([
            [1, 0, 0, 0],
            [0, 1, 0, 0],
        ], np.float32)

        self.kf.processNoiseCov = np.eye(4, dtype=np.float32) * 0.05
        self.kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * 3.0
        self.kf.errorCovPost = np.eye(4, dtype=np.float32)

        x, y = xy
        self.kf.statePost = np.array([[x], [y], [0], [0]], np.float32)

        self.radius = float(radius)
        self.hist = hist
        self.quality = float(quality)

        self.missed = 0
        self.age = 1
        self.hits = 1

        self.trace = [(float(x), float(y))]
        self.last_velocity = np.array([0.0, 0.0], dtype=np.float32)

    def predict(self):
        p = self.kf.predict()
        return np.array([p[0, 0], p[1, 0]], dtype=np.float32)

    def current_position(self):
        return np.array(self.trace[-1], dtype=np.float32)

    def update(self, det):
        old_pos = self.current_position()
        new_pos = np.array([det["x"], det["y"]], dtype=np.float32)

        self.last_velocity = new_pos - old_pos

        measurement = np.array([
            [np.float32(det["x"])],
            [np.float32(det["y"])]
        ])

        self.kf.correct(measurement)

        self.radius = 0.85 * self.radius + 0.15 * det["r"]
        self.quality = 0.85 * self.quality + 0.15 * det["quality"]

        if det["hist"] is not None:
            if self.hist is None:
                self.hist = det["hist"]
            else:
                self.hist = 0.9 * self.hist + 0.1 * det["hist"]

        self.missed = 0
        self.age += 1
        self.hits += 1
        self.trace.append((float(det["x"]), float(det["y"])))

    def mark_missed(self):
        pred = self.predict()
        self.missed += 1
        self.age += 1
        self.trace.append((float(pred[0]), float(pred[1])))


# ----------------------------
# Assignment
# ----------------------------

def near_collision(track_idx, predicted_positions, avg_radius):
    p = predicted_positions[track_idx]

    for k, q in enumerate(predicted_positions):
        if k == track_idx:
            continue

        if np.linalg.norm(p - q) < 2.5 * avg_radius:
            return True

    return False


def assign_tracks(tracks, detections):
    if len(tracks) == 0:
        return [], list(range(len(detections))), []

    if len(detections) == 0:
        return [], [], list(range(len(tracks)))

    predicted = np.array([t.predict() for t in tracks], dtype=np.float32)
    avg_radius = np.mean([t.radius for t in tracks]) if tracks else 12.0

    cost = np.full((len(tracks), len(detections)), 1e6, dtype=np.float32)

    for i, t in enumerate(tracks):
        collision_mode = near_collision(i, predicted, avg_radius)

        for j, det in enumerate(detections):
            det_pos = np.array([det["x"], det["y"]], dtype=np.float32)

            if det["quality"] < MIN_ASSIGN_QUALITY:
                continue

            jump_dist = np.linalg.norm(predicted[i] - det_pos)
            max_jump = max(
                BASE_MAX_JUMP,
                VELOCITY_JUMP_FACTOR * np.linalg.norm(t.last_velocity) + JUMP_EXTRA
            )

            if jump_dist > max_jump:
                continue

            position_cost = min(jump_dist / max_jump, 1.0)

            color_cost = hist_cost(t.hist, det["hist"])

            radius_cost = abs(t.radius - det["r"]) / max(t.radius, det["r"], 1.0)
            radius_cost = min(radius_cost, 1.0)

            expected_pos = t.current_position() + t.last_velocity
            velocity_cost = np.linalg.norm(expected_pos - det_pos) / max(max_jump, 1.0)
            velocity_cost = min(velocity_cost, 1.0)

            quality_cost = 1.0 - det["quality"]

            if collision_mode:
                cost[i, j] = (
                    0.30 * position_cost +
                    0.40 * color_cost +
                    0.10 * radius_cost +
                    0.10 * velocity_cost +
                    0.10 * quality_cost
                )
            else:
                cost[i, j] = (
                    0.45 * position_cost +
                    0.25 * color_cost +
                    0.10 * radius_cost +
                    0.10 * velocity_cost +
                    0.10 * quality_cost
                )

    row_ind, col_ind = linear_sum_assignment(cost)

    matches = []
    unmatched_dets = set(range(len(detections)))
    unmatched_tracks = set(range(len(tracks)))

    for r, c in zip(row_ind, col_ind):
        if cost[r, c] <= MAX_ASSIGN_COST:
            matches.append((r, c))
            unmatched_tracks.discard(r)
            unmatched_dets.discard(c)

    return matches, list(unmatched_dets), list(unmatched_tracks)


# ----------------------------
# Pending detections
# ----------------------------

def update_pending(pending, unmatched_dets, detections):
    used = set()

    for p in pending:
        best_i = None
        best_d = 1e9

        for di in unmatched_dets:
            if di in used:
                continue

            det = detections[di]
            d = np.linalg.norm(
                p["xy"] - np.array([det["x"], det["y"]], dtype=np.float32)
            )

            if d < best_d:
                best_d = d
                best_i = di

        if best_i is not None and best_d < 25:
            det = detections[best_i]
            p["xy"] = np.array([det["x"], det["y"]], dtype=np.float32)
            p["r"] = det["r"]
            p["quality"] = det["quality"]
            p["hist"] = det["hist"]
            p["age"] += 1
            p["hits"] += 1
            used.add(best_i)
        else:
            p["age"] += 1

    for di in unmatched_dets:
        if di not in used:
            det = detections[di]
            pending.append({
                "xy": np.array([det["x"], det["y"]], dtype=np.float32),
                "r": det["r"],
                "quality": det["quality"],
                "hist": det["hist"],
                "age": 1,
                "hits": 1,
            })

    promoted = [
        p for p in pending
        if p["hits"] >= PENDING_HITS_TO_PROMOTE
    ]

    pending[:] = [
        p for p in pending
        if p["age"] <= PENDING_MAX_AGE and p["hits"] < PENDING_HITS_TO_PROMOTE
    ]

    return promoted


# ----------------------------
# Drawing
# ----------------------------

def id_color(track_id):
    rng = np.random.default_rng(track_id + 12345)
    return tuple(int(v) for v in rng.integers(60, 255, size=3))


def draw_tracks(frame, tracks):
    for t in tracks:
        color = id_color(t.id)

        pts = np.array(t.trace[-TRACE_LENGTH:], dtype=np.int32)

        if len(pts) >= 2:
            for i in range(1, len(pts)):
                cv2.line(frame, tuple(pts[i - 1]), tuple(pts[i]), color, 2)

        x, y = pts[-1]
        cv2.circle(frame, (x, y), int(round(t.radius)), color, 2)
        cv2.circle(frame, (x, y), 3, color, -1)

        cv2.putText(
            frame,
            f"ID {t.id}",
            (x + 8, y - 8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            color,
            1,
            cv2.LINE_AA,
        )


def draw_detections(frame, detections):
    for det in detections:
        x, y, r = int(det["x"]), int(det["y"]), int(det["r"])
        q = det["quality"]

        cv2.circle(frame, (x, y), r, (0, 255, 0), 1)
        cv2.putText(
            frame,
            f"{q:.2f}",
            (x - 10, y + r + 12),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.35,
            (0, 255, 0),
            1,
            cv2.LINE_AA,
        )


# ----------------------------
# Main
# ----------------------------

def main():
    cap = cv2.VideoCapture(VIDEO)

    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {VIDEO}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 30.0

    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    writer = cv2.VideoWriter(
        OUT,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (w, h),
    )

    tracks = []
    pending = []
    next_id = 0
    frame_idx = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        detections = detect_circles(frame)

        matches, unmatched_dets, unmatched_tracks = assign_tracks(
            tracks,
            detections
        )

        for ti, di in matches:
            tracks[ti].update(detections[di])

        for ti in unmatched_tracks:
            tracks[ti].mark_missed()

        promoted = update_pending(
            pending,
            unmatched_dets,
            detections
        )

        for p in promoted:
            x, y = p["xy"]
            tracks.append(
                Track(
                    xy=(x, y),
                    radius=p["r"],
                    hist=p["hist"],
                    quality=p["quality"],
                    track_id=next_id
                )
            )
            next_id += 1

        tracks = [t for t in tracks if t.missed <= MAX_MISSED]

        vis = frame.copy()

        # table margin visualization
        cv2.rectangle(
            vis,
            (TABLE_MARGIN, TABLE_MARGIN),
            (w - TABLE_MARGIN, h - TABLE_MARGIN),
            (255, 0, 0),
            1,
        )

        draw_detections(vis, detections)
        draw_tracks(vis, tracks)

        cv2.putText(
            vis,
            f"frame {frame_idx} | detections {len(detections)} | tracks {len(tracks)} | pending {len(pending)}",
            (20, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2,
            cv2.LINE_AA,
        )

        writer.write(vis)
        frame_idx += 1

    cap.release()
    writer.release()

    print(f"Saved: {OUT}")


if __name__ == "__main__":
    main()

Saved: tracked_hough_v2.mp4


In [2]:
import cv2
import numpy as np
import sys
from scipy.optimize import linear_sum_assignment


VIDEO = "Houston, we Have an Overhead!  8 Ball Match Play!.mp4"
OUT = "tracked_hough.mp4"


# ----------------------------
# ROI: edit these
# ----------------------------
ROI_X1 = 120
ROI_Y1 = 80
ROI_X2 = 1800
ROI_Y2 = 920

DRAW_ROI = True


class Track:
    def __init__(self, xy, radius, hist, track_id):
        self.id = track_id
        self.kf = cv2.KalmanFilter(4, 2)
        self.kf.transitionMatrix = np.array([
            [1, 0, 1, 0],
            [0, 1, 0, 1],
            [0, 0, 1, 0],
            [0, 0, 0, 1],
        ], np.float32)
        self.kf.measurementMatrix = np.array([
            [1, 0, 0, 0],
            [0, 1, 0, 0],
        ], np.float32)

        self.kf.processNoiseCov = np.eye(4, dtype=np.float32) * 0.03
        self.kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * 3.0
        self.kf.errorCovPost = np.eye(4, dtype=np.float32)

        x, y = xy
        self.kf.statePost = np.array([[x], [y], [0], [0]], np.float32)

        self.missed = 0
        self.trace = [(float(x), float(y))]
        self.radius = radius
        self.hist = hist
        self.last_velocity = np.array([0.0, 0.0])

    def predict(self):
        p = self.kf.predict()
        return np.array([p[0, 0], p[1, 0]])

    def update(self, xy, radius, hist):
        old_pos = np.array(self.trace[-1])
        new_pos = np.array(xy)

        self.last_velocity = new_pos - old_pos
        self.radius = 0.8 * self.radius + 0.2 * radius

        if hist is not None:
            if self.hist is None:
                self.hist = hist
            else:
                self.hist = 0.9 * self.hist + 0.1 * hist

        self.kf.correct(np.array([[np.float32(xy[0])], [np.float32(xy[1])]]))
        self.missed = 0
        self.trace.append((float(xy[0]), float(xy[1])))

    def mark_missed(self):
        self.missed += 1
        p = self.predict()
        self.trace.append((float(p[0]), float(p[1])))


def get_roi(frame):
    h, w = frame.shape[:2]

    x1 = max(0, min(ROI_X1, w - 1))
    y1 = max(0, min(ROI_Y1, h - 1))
    x2 = max(0, min(ROI_X2, w))
    y2 = max(0, min(ROI_Y2, h))

    if x2 <= x1 or y2 <= y1:
        raise ValueError("Invalid ROI coordinates")

    roi = frame[y1:y2, x1:x2]
    return roi, x1, y1, x2, y2


def detect_circles(frame):
    """
    Detect only inside ROI, then convert circle centers
    back to full-frame coordinates.
    """
    roi, ox, oy, _, _ = get_roi(frame)

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    blur = cv2.medianBlur(gray, 5)

    circles = cv2.HoughCircles(
        blur,
        cv2.HOUGH_GRADIENT,
        dp=1.2,
        minDist=22,
        param1=80,
        param2=18,
        minRadius=6,
        maxRadius=22,
    )

    detections = []

    if circles is not None:
        circles = np.round(circles[0]).astype(int)

        for x, y, r in circles:
            # Reject detections near ROI border
            if x < 5 or y < 5 or x >= roi.shape[1] - 5 or y >= roi.shape[0] - 5:
                continue

            # Convert ROI-local coords to full-frame coords
            full_x = x + ox
            full_y = y + oy

            detections.append((float(full_x), float(full_y), float(r)))

    return detections


def assign_tracks(tracks, detections, frame, max_cost=0.75):
    if not tracks:
        return [], list(range(len(detections))), []

    if not detections:
        return [], [], list(range(len(tracks)))

    cost = np.zeros((len(tracks), len(detections)), dtype=np.float32)

    det_hists = []
    for x, y, r in detections:
        det_hists.append(circle_histogram(frame, x, y, r))

    for i, t in enumerate(tracks):
        pred = t.predict()

        for j, (x, y, r) in enumerate(detections):
            det_pos = np.array([x, y])

            position_cost = np.linalg.norm(pred - det_pos) / 60.0
            position_cost = min(position_cost, 1.0)

            color_cost = hist_cost(t.hist, det_hists[j])

            radius_cost = abs(t.radius - r) / max(t.radius, r, 1.0)
            radius_cost = min(radius_cost, 1.0)

            expected_pos = np.array(t.trace[-1]) + t.last_velocity
            velocity_cost = np.linalg.norm(expected_pos - det_pos) / 80.0
            velocity_cost = min(velocity_cost, 1.0)

            cost[i, j] = (
                0.55 * position_cost +
                0.30 * color_cost +
                0.10 * radius_cost +
                0.05 * velocity_cost
            )

    row_ind, col_ind = linear_sum_assignment(cost)

    matches = []
    unmatched_dets = set(range(len(detections)))
    unmatched_tracks = set(range(len(tracks)))

    for r, c in zip(row_ind, col_ind):
        if cost[r, c] <= max_cost:
            matches.append((r, c, det_hists[c]))
            unmatched_tracks.discard(r)
            unmatched_dets.discard(c)

    return matches, list(unmatched_dets), list(unmatched_tracks)


def draw_tracks(frame, tracks):
    for t in tracks:
        if len(t.trace) < 2:
            continue

        pts = np.array(t.trace[-80:], dtype=np.int32)

        for i in range(1, len(pts)):
            cv2.line(frame, tuple(pts[i - 1]), tuple(pts[i]), (0, 255, 255), 2)

        x, y = pts[-1]
        cv2.circle(frame, (x, y), 5, (0, 0, 255), -1)
        cv2.putText(
            frame,
            f"ID {t.id}",
            (x + 8, y - 8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (255, 255, 255),
            1,
        )


def circle_histogram(frame, x, y, r):
    h, w = frame.shape[:2]

    x, y, r = int(x), int(y), int(r)
    x1, x2 = max(0, x - r), min(w, x + r)
    y1, y2 = max(0, y - r), min(h, y + r)

    patch = frame[y1:y2, x1:x2]
    if patch.size == 0:
        return None

    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)

    mask = np.zeros((y2 - y1, x2 - x1), dtype=np.uint8)
    cv2.circle(mask, (x - x1, y - y1), r, 255, -1)

    hist = cv2.calcHist(
        [hsv],
        [0, 1],
        mask,
        [24, 16],
        [0, 180, 0, 256]
    )

    cv2.normalize(hist, hist)
    return hist


def hist_cost(h1, h2):
    if h1 is None or h2 is None:
        return 1.0

    similarity = cv2.compareHist(h1, h2, cv2.HISTCMP_CORREL)
    return 1.0 - max(0.0, similarity)


cap = cv2.VideoCapture(VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer = cv2.VideoWriter(
    OUT,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w, h),
)

tracks = []
next_id = 0
max_missed = 12

frame_idx = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    detections = detect_circles(frame)

    matches, unmatched_dets, unmatched_tracks = assign_tracks(
        tracks, detections, frame
    )

    for ti, di, appearance in matches:
        x, y, r = detections[di]
        tracks[ti].update((x, y), r, appearance)

    for ti in unmatched_tracks:
        tracks[ti].mark_missed()

    for di in unmatched_dets:
        x, y, r = detections[di]
        appearance = circle_histogram(frame, x, y, r)
        tracks.append(Track((x, y), r, appearance, next_id))
        next_id += 1

    tracks = [t for t in tracks if t.missed <= max_missed]

    if DRAW_ROI:
        cv2.rectangle(
            frame,
            (ROI_X1, ROI_Y1),
            (ROI_X2, ROI_Y2),
            (255, 0, 0),
            2,
        )

    for x, y, r in detections:
        cv2.circle(frame, (int(x), int(y)), int(r), (0, 255, 0), 2)

    draw_tracks(frame, tracks)

    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()

print(f"Saved: {OUT}")

Saved: tracked_hough.mp4


In [1]:
import cv2
import numpy as np
from scipy.optimize import linear_sum_assignment


VIDEO = "Houston, we Have an Overhead!  8 Ball Match Play!.mp4"
OUT = "tracked_hough.mp4"


# ----------------------------
# ROI: edit these
# ----------------------------
ROI_X1 = 120
ROI_Y1 = 80
ROI_X2 = 1800
ROI_Y2 = 920

DRAW_ROI = True


class Track:
    def __init__(self, xy, radius, appearance, track_id):
        self.id = track_id
        self.kf = cv2.KalmanFilter(4, 2)

        self.kf.transitionMatrix = np.array([
            [1, 0, 1, 0],
            [0, 1, 0, 1],
            [0, 0, 1, 0],
            [0, 0, 0, 1],
        ], np.float32)

        self.kf.measurementMatrix = np.array([
            [1, 0, 0, 0],
            [0, 1, 0, 0],
        ], np.float32)

        self.kf.processNoiseCov = np.eye(4, dtype=np.float32) * 0.03
        self.kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * 3.0
        self.kf.errorCovPost = np.eye(4, dtype=np.float32)

        x, y = xy
        self.kf.statePost = np.array([[x], [y], [0], [0]], np.float32)

        self.missed = 0
        self.trace = [(float(x), float(y))]
        self.radius = radius
        self.appearance = appearance
        self.last_velocity = np.array([0.0, 0.0])

    def predict(self):
        p = self.kf.predict()
        return np.array([p[0, 0], p[1, 0]])

    def update(self, xy, radius, appearance):
        old_pos = np.array(self.trace[-1])
        new_pos = np.array(xy)

        self.last_velocity = new_pos - old_pos
        self.radius = 0.8 * self.radius + 0.2 * radius

        if appearance is not None:
            if self.appearance is None:
                self.appearance = appearance
            else:
                self.appearance["hue_hist"] = (
                    0.97 * self.appearance["hue_hist"] +
                    0.03 * appearance["hue_hist"]
                )

                self.appearance["white_frac"] = (
                    0.98 * self.appearance["white_frac"] +
                    0.02 * appearance["white_frac"]
                )

                self.appearance["color_frac"] = (
                    0.98 * self.appearance["color_frac"] +
                    0.02 * appearance["color_frac"]
                )

        self.kf.correct(np.array([[np.float32(xy[0])], [np.float32(xy[1])]]))
        self.missed = 0
        self.trace.append((float(xy[0]), float(xy[1])))

    def mark_missed(self):
        self.missed += 1
        p = self.predict()
        self.trace.append((float(p[0]), float(p[1])))


def get_roi(frame):
    h, w = frame.shape[:2]

    x1 = max(0, min(ROI_X1, w - 1))
    y1 = max(0, min(ROI_Y1, h - 1))
    x2 = max(0, min(ROI_X2, w))
    y2 = max(0, min(ROI_Y2, h))

    if x2 <= x1 or y2 <= y1:
        raise ValueError("Invalid ROI coordinates")

    roi = frame[y1:y2, x1:x2]
    return roi, x1, y1, x2, y2


def detect_circles(frame):
    roi, ox, oy, _, _ = get_roi(frame)

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    blur = cv2.medianBlur(gray, 5)

    circles = cv2.HoughCircles(
        blur,
        cv2.HOUGH_GRADIENT,
        dp=1.2,
        minDist=22,
        param1=80,
        param2=18,
        minRadius=6,
        maxRadius=22,
    )

    detections = []

    if circles is not None:
        circles = np.round(circles[0]).astype(int)

        for x, y, r in circles:
            if x < 5 or y < 5 or x >= roi.shape[1] - 5 or y >= roi.shape[0] - 5:
                continue

            full_x = x + ox
            full_y = y + oy

            detections.append((float(full_x), float(full_y), float(r)))

    return detections


def ball_appearance(frame, x, y, r):
    h, w = frame.shape[:2]

    x, y, r = int(x), int(y), int(r)

    x1, x2 = max(0, x - r), min(w, x + r)
    y1, y2 = max(0, y - r), min(h, y + r)

    patch = frame[y1:y2, x1:x2]
    if patch.size == 0:
        return None

    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)

    mask = np.zeros((y2 - y1, x2 - x1), dtype=np.uint8)

    cx = x - x1
    cy = y - y1

    safe_r = min(
        r,
        cx,
        cy,
        mask.shape[1] - cx - 1,
        mask.shape[0] - cy - 1,
    )

    if safe_r <= 2:
        return None

    cv2.circle(mask, (cx, cy), safe_r, 255, -1)

    h_ch = hsv[:, :, 0]
    s_ch = hsv[:, :, 1]
    v_ch = hsv[:, :, 2]

    ball_pixels = mask > 0

    white_pixels = ball_pixels & (s_ch < 45) & (v_ch > 120)
    color_pixels = ball_pixels & (s_ch > 55) & (v_ch > 50)

    total = max(np.count_nonzero(ball_pixels), 1)
    white_frac = np.count_nonzero(white_pixels) / total
    color_frac = np.count_nonzero(color_pixels) / total

    if np.count_nonzero(color_pixels) < 8:
        hue_hist = np.zeros((18, 1), dtype=np.float32)
    else:
        hue_hist = cv2.calcHist(
            [h_ch],
            [0],
            color_pixels.astype(np.uint8),
            [18],
            [0, 180]
        )
        cv2.normalize(hue_hist, hue_hist)

    return {
        "hue_hist": hue_hist.astype(np.float32),
        "white_frac": float(white_frac),
        "color_frac": float(color_frac),
    }


def appearance_cost(a1, a2):
    if a1 is None or a2 is None:
        return 0.5

    h1 = a1["hue_hist"]
    h2 = a2["hue_hist"]

    if np.count_nonzero(h1) == 0 or np.count_nonzero(h2) == 0:
        hue_cost = 0.5
    else:
        hue_sim = cv2.compareHist(h1, h2, cv2.HISTCMP_CORREL)

        if np.isnan(hue_sim):
            hue_cost = 0.5
        else:
            hue_cost = 1.0 - max(0.0, min(1.0, hue_sim))

    white_cost = abs(a1["white_frac"] - a2["white_frac"])
    color_cost = abs(a1["color_frac"] - a2["color_frac"])

    return (
        0.75 * hue_cost +
        0.15 * white_cost +
        0.10 * color_cost
    )


def assign_tracks(tracks, detections, frame, max_cost=0.75):
    if not tracks:
        return [], list(range(len(detections))), []

    if not detections:
        return [], [], list(range(len(tracks)))

    cost = np.zeros((len(tracks), len(detections)), dtype=np.float32)

    det_apps = []
    for x, y, r in detections:
        det_apps.append(ball_appearance(frame, x, y, r))

    for i, t in enumerate(tracks):
        pred = t.predict()

        for j, (x, y, r) in enumerate(detections):
            det_pos = np.array([x, y])

            position_cost = np.linalg.norm(pred - det_pos) / 60.0
            position_cost = min(position_cost, 1.0)

            color_cost = appearance_cost(t.appearance, det_apps[j])

            radius_cost = abs(t.radius - r) / max(t.radius, r, 1.0)
            radius_cost = min(radius_cost, 1.0)

            expected_pos = np.array(t.trace[-1]) + t.last_velocity
            velocity_cost = np.linalg.norm(expected_pos - det_pos) / 80.0
            velocity_cost = min(velocity_cost, 1.0)

            cost[i, j] = (
                0.65 * position_cost +
                0.20 * color_cost +
                0.10 * radius_cost +
                0.05 * velocity_cost
            )

    row_ind, col_ind = linear_sum_assignment(cost)

    matches = []
    unmatched_dets = set(range(len(detections)))
    unmatched_tracks = set(range(len(tracks)))

    for r, c in zip(row_ind, col_ind):
        if cost[r, c] <= max_cost:
            matches.append((r, c, det_apps[c]))
            unmatched_tracks.discard(r)
            unmatched_dets.discard(c)

    return matches, list(unmatched_dets), list(unmatched_tracks)


def draw_tracks(frame, tracks):
    for t in tracks:
        if len(t.trace) < 2:
            continue

        pts = np.array(t.trace[-80:], dtype=np.int32)

        for i in range(1, len(pts)):
            cv2.line(frame, tuple(pts[i - 1]), tuple(pts[i]), (0, 255, 255), 2)

        x, y = pts[-1]
        cv2.circle(frame, (x, y), 5, (0, 0, 255), -1)

        cv2.putText(
            frame,
            f"ID {t.id}",
            (x + 8, y - 8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (255, 255, 255),
            1,
        )


cap = cv2.VideoCapture(VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS)

w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer = cv2.VideoWriter(
    OUT,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w, h),
)

tracks = []
next_id = 0
max_missed = 12
frame_idx = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    detections = detect_circles(frame)

    matches, unmatched_dets, unmatched_tracks = assign_tracks(
        tracks,
        detections,
        frame
    )

    for ti, di, app in matches:
        x, y, r = detections[di]
        tracks[ti].update((x, y), r, app)

    for ti in unmatched_tracks:
        tracks[ti].mark_missed()

    for di in unmatched_dets:
        x, y, r = detections[di]
        app = ball_appearance(frame, x, y, r)
        tracks.append(Track((x, y), r, app, next_id))
        next_id += 1

    tracks = [t for t in tracks if t.missed <= max_missed]

    if DRAW_ROI:
        cv2.rectangle(
            frame,
            (ROI_X1, ROI_Y1),
            (ROI_X2, ROI_Y2),
            (255, 0, 0),
            2,
        )

    for x, y, r in detections:
        cv2.circle(frame, (int(x), int(y)), int(r), (0, 255, 0), 2)

    draw_tracks(frame, tracks)

    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()

print(f"Saved: {OUT}")

Saved: tracked_hough.mp4


In [2]:
import cv2
import numpy as np
from scipy.optimize import linear_sum_assignment


VIDEO = "Houston, we Have an Overhead!  8 Ball Match Play!.mp4"
OUT = "tracked_hough_hsv.mp4"


ROI_X1 = 120
ROI_Y1 = 80
ROI_X2 = 1800
ROI_Y2 = 920

DRAW_ROI = True


class Track:
    def __init__(self, xy, radius, hsv_signature, track_id):
        self.id = track_id

        self.kf = cv2.KalmanFilter(4, 2)
        self.kf.transitionMatrix = np.array([
            [1, 0, 1, 0],
            [0, 1, 0, 1],
            [0, 0, 1, 0],
            [0, 0, 0, 1],
        ], np.float32)

        self.kf.measurementMatrix = np.array([
            [1, 0, 0, 0],
            [0, 1, 0, 0],
        ], np.float32)

        self.kf.processNoiseCov = np.eye(4, dtype=np.float32) * 0.03
        self.kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * 3.0
        self.kf.errorCovPost = np.eye(4, dtype=np.float32)

        x, y = xy
        self.kf.statePost = np.array([[x], [y], [0], [0]], np.float32)

        self.trace = [(float(x), float(y))]
        self.radius = float(radius)
        self.hsv_signature = hsv_signature
        self.last_velocity = np.array([0.0, 0.0])
        self.missed = 0

    def predict(self):
        p = self.kf.predict()
        return np.array([p[0, 0], p[1, 0]])

    def update(self, xy, radius, hsv_signature):
        old_pos = np.array(self.trace[-1])
        new_pos = np.array(xy)

        self.last_velocity = new_pos - old_pos
        self.radius = 0.8 * self.radius + 0.2 * radius

        if hsv_signature is not None:
            if self.hsv_signature is None:
                self.hsv_signature = hsv_signature
            else:
                self.hsv_signature["hue_hist"] = (
                    0.97 * self.hsv_signature["hue_hist"] +
                    0.03 * hsv_signature["hue_hist"]
                )

                self.hsv_signature["sat_hist"] = (
                    0.97 * self.hsv_signature["sat_hist"] +
                    0.03 * hsv_signature["sat_hist"]
                )

                self.hsv_signature["white_frac"] = (
                    0.98 * self.hsv_signature["white_frac"] +
                    0.02 * hsv_signature["white_frac"]
                )

                self.hsv_signature["color_frac"] = (
                    0.98 * self.hsv_signature["color_frac"] +
                    0.02 * hsv_signature["color_frac"]
                )

        self.kf.correct(np.array([[np.float32(xy[0])], [np.float32(xy[1])]]))
        self.missed = 0
        self.trace.append((float(xy[0]), float(xy[1])))

    def mark_missed(self):
        self.missed += 1
        p = self.predict()
        self.trace.append((float(p[0]), float(p[1])))


def get_roi(frame):
    h, w = frame.shape[:2]

    x1 = max(0, min(ROI_X1, w - 1))
    y1 = max(0, min(ROI_Y1, h - 1))
    x2 = max(0, min(ROI_X2, w))
    y2 = max(0, min(ROI_Y2, h))

    roi = frame[y1:y2, x1:x2]
    return roi, x1, y1, x2, y2


def detect_circles(frame):
    roi, ox, oy, _, _ = get_roi(frame)

    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    h_ch, s_ch, v_ch = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]

    # Approx green cloth mask. Tune these.
    green_mask = (
        (h_ch > 35) & (h_ch < 95) &
        (s_ch > 40) &
        (v_ch > 40)
    )

    # Candidate ball image: bright/saturated things that are NOT table cloth
    candidate = np.zeros_like(v_ch)
    candidate[~green_mask] = v_ch[~green_mask]

    candidate = cv2.equalizeHist(candidate)
    blur = cv2.medianBlur(candidate, 5)

    circles = cv2.HoughCircles(
        blur,
        cv2.HOUGH_GRADIENT,
        dp=1.2,
        minDist=22,
        param1=80,
        param2=16,
        minRadius=6,
        maxRadius=22,
    )

    detections = []

    if circles is not None:
        circles = np.round(circles[0]).astype(int)

        for x, y, r in circles:
            if x < 5 or y < 5 or x >= roi.shape[1] - 5 or y >= roi.shape[0] - 5:
                continue

            detections.append((float(x + ox), float(y + oy), float(r)))

    return detections


def hsv_ball_signature(frame, x, y, r):
    h, w = frame.shape[:2]

    x, y, r = int(x), int(y), int(r)
    x1, x2 = max(0, x - r), min(w, x + r)
    y1, y2 = max(0, y - r), min(h, y + r)

    patch = frame[y1:y2, x1:x2]
    if patch.size == 0:
        return None

    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)

    mask = np.zeros((y2 - y1, x2 - x1), dtype=np.uint8)
    cx = x - x1
    cy = y - y1

    safe_r = min(
        r,
        cx,
        cy,
        mask.shape[1] - cx - 1,
        mask.shape[0] - cy - 1,
    )

    if safe_r <= 2:
        return None

    cv2.circle(mask, (cx, cy), safe_r, 255, -1)

    H = hsv[:, :, 0]
    S = hsv[:, :, 1]
    V = hsv[:, :, 2]

    ball_pixels = mask > 0

    white_pixels = ball_pixels & (S < 45) & (V > 120)
    colored_pixels = ball_pixels & (S > 55) & (V > 50)

    total = max(np.count_nonzero(ball_pixels), 1)
    white_frac = np.count_nonzero(white_pixels) / total
    color_frac = np.count_nonzero(colored_pixels) / total

    if np.count_nonzero(colored_pixels) >= 8:
        hue_hist = cv2.calcHist(
            [H],
            [0],
            colored_pixels.astype(np.uint8),
            [24],
            [0, 180],
        )
        cv2.normalize(hue_hist, hue_hist)
    else:
        hue_hist = np.zeros((24, 1), dtype=np.float32)

    sat_hist = cv2.calcHist(
        [S],
        [0],
        mask,
        [16],
        [0, 256],
    )
    cv2.normalize(sat_hist, sat_hist)

    return {
        "hue_hist": hue_hist.astype(np.float32),
        "sat_hist": sat_hist.astype(np.float32),
        "white_frac": float(white_frac),
        "color_frac": float(color_frac),
    }


def hist_distance(h1, h2):
    if h1 is None or h2 is None:
        return 0.5

    if np.count_nonzero(h1) == 0 or np.count_nonzero(h2) == 0:
        return 0.5

    sim = cv2.compareHist(h1, h2, cv2.HISTCMP_CORREL)

    if np.isnan(sim):
        return 0.5

    return 1.0 - max(0.0, min(1.0, sim))


def hsv_signature_cost(a, b):
    if a is None or b is None:
        return 0.5

    hue_cost = hist_distance(a["hue_hist"], b["hue_hist"])
    sat_cost = hist_distance(a["sat_hist"], b["sat_hist"])

    # Stripe balls rotate, so white/color fractions are useful but weak.
    white_cost = abs(a["white_frac"] - b["white_frac"])
    color_cost = abs(a["color_frac"] - b["color_frac"])

    return (
        0.70 * hue_cost +
        0.15 * sat_cost +
        0.10 * white_cost +
        0.05 * color_cost
    )


def assign_tracks(tracks, detections, frame, max_cost=0.75):
    if not tracks:
        return [], list(range(len(detections))), []

    if not detections:
        return [], [], list(range(len(tracks)))

    det_signatures = [
        hsv_ball_signature(frame, x, y, r)
        for x, y, r in detections
    ]

    cost = np.zeros((len(tracks), len(detections)), dtype=np.float32)

    for i, t in enumerate(tracks):
        pred = t.predict()

        for j, (x, y, r) in enumerate(detections):
            det_pos = np.array([x, y])

            position_cost = np.linalg.norm(pred - det_pos) / 60.0
            position_cost = min(position_cost, 1.0)

            hsv_cost = hsv_signature_cost(
                t.hsv_signature,
                det_signatures[j]
            )

            radius_cost = abs(t.radius - r) / max(t.radius, r, 1.0)
            radius_cost = min(radius_cost, 1.0)

            expected_pos = np.array(t.trace[-1]) + t.last_velocity
            velocity_cost = np.linalg.norm(expected_pos - det_pos) / 80.0
            velocity_cost = min(velocity_cost, 1.0)

            # Motion dominates; HSV helps prevent ID swaps.
            cost[i, j] = (
                0.65 * position_cost +
                0.20 * hsv_cost +
                0.10 * radius_cost +
                0.05 * velocity_cost
            )

    row_ind, col_ind = linear_sum_assignment(cost)

    matches = []
    unmatched_dets = set(range(len(detections)))
    unmatched_tracks = set(range(len(tracks)))

    for r, c in zip(row_ind, col_ind):
        if cost[r, c] <= max_cost:
            matches.append((r, c, det_signatures[c]))
            unmatched_tracks.discard(r)
            unmatched_dets.discard(c)

    return matches, list(unmatched_dets), list(unmatched_tracks)


def draw_tracks(frame, tracks):
    for t in tracks:
        if len(t.trace) < 2:
            continue

        pts = np.array(t.trace[-80:], dtype=np.int32)

        for i in range(1, len(pts)):
            cv2.line(frame, tuple(pts[i - 1]), tuple(pts[i]), (0, 255, 255), 2)

        x, y = pts[-1]

        cv2.circle(frame, (x, y), 5, (0, 0, 255), -1)
        cv2.putText(
            frame,
            f"ID {t.id}",
            (x + 8, y - 8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (255, 255, 255),
            1,
        )


cap = cv2.VideoCapture(VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS)

w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer = cv2.VideoWriter(
    OUT,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w, h),
)

tracks = []
next_id = 0
max_missed = 12
frame_idx = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    detections = detect_circles(frame)

    matches, unmatched_dets, unmatched_tracks = assign_tracks(
        tracks,
        detections,
        frame
    )

    for ti, di, sig in matches:
        x, y, r = detections[di]
        tracks[ti].update((x, y), r, sig)

    for ti in unmatched_tracks:
        tracks[ti].mark_missed()

    for di in unmatched_dets:
        x, y, r = detections[di]
        sig = hsv_ball_signature(frame, x, y, r)
        tracks.append(Track((x, y), r, sig, next_id))
        next_id += 1

    tracks = [t for t in tracks if t.missed <= max_missed]

    if DRAW_ROI:
        cv2.rectangle(
            frame,
            (ROI_X1, ROI_Y1),
            (ROI_X2, ROI_Y2),
            (255, 0, 0),
            2,
        )

    for x, y, r in detections:
        cv2.circle(frame, (int(x), int(y)), int(r), (0, 255, 0), 2)

    draw_tracks(frame, tracks)

    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()

print(f"Saved: {OUT}")

KeyboardInterrupt: 

In [3]:

cap.release()
writer.release()